# 02 - Generate OSS Model Responses

            This notebook loads the prompt dataset, runs open-source causal language models, autosaves partial generations to Google Drive, logs progress to W&B, and publishes the combined generations dataset to Hugging Face.

            Default models are Qwen 0.5B/1.5B base and instruct checkpoints. Gemma 2B models are included as optional entries because they may require accepting the model license on Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", "")
    if repo_url:
        target = Path("/content/empty-negations")
        if not target.exists():
            subprocess.run(["git", "clone", repo_url, str(target)], check=True)
        return target
    raise FileNotFoundError(
        "Could not find the empty-negations repo. Run this notebook from the repo, "
        "copy it to /content/drive/MyDrive/ocn_empty_negations, or set OCN_REPO_URL."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import gc, json
            from pathlib import Path
            import pandas as pd
            import torch
            import wandb
            from datasets import load_dataset

            from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe
            from ocn.generation import (
                DecodingSpec,
                ModelSpec,
                generation_rows,
                load_text_generation_model,
            )

            paths = make_colab_paths()
            config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
            login_huggingface("HF_WRITE_ACCESS")
            run = login_wandb(project="ocn-empty-negations", name=f"generate-{config['run_id']}", config=config)

In [ ]:
prompts = load_dataset(config["hf_prompt_repo"], split="train").to_pandas()

            RUN_MODE = "pilot"  # change to "main" for the full prompt bank
            PROMPT_LIMIT = config["default_prompt_limit"] if RUN_MODE == "pilot" else None
            if PROMPT_LIMIT:
                prompts = prompts.sample(n=min(PROMPT_LIMIT, len(prompts)), random_state=13).sort_values("prompt_id")

            MODEL_SPECS = [
                ModelSpec("Qwen/Qwen2.5-0.5B", "qwen", "base", False),
                ModelSpec("Qwen/Qwen2.5-0.5B-Instruct", "qwen", "instruct", True),
                ModelSpec("Qwen/Qwen2.5-1.5B-Instruct", "qwen", "instruct", True),
                # Uncomment after accepting Gemma terms on Hugging Face:
                # ModelSpec("google/gemma-2-2b", "gemma", "base", False),
                # ModelSpec("google/gemma-2-2b-it", "gemma", "instruct", True),
            ]

            DECODINGS = [
                DecodingSpec("greedy", temperature=0.0, top_p=1.0, max_new_tokens=160),
                DecodingSpec("normal_temp", temperature=0.7, top_p=0.95, max_new_tokens=180),
            ]

            SEEDS = config["default_seeds"]
            QUANTIZE_4BIT = True
            print("Prompts:", len(prompts), "Models:", len(MODEL_SPECS), "Decodings:", len(DECODINGS), "Seeds:", SEEDS)

In [ ]:
all_parts = []
            generation_dir = Path(config["drive_data_root"]) / "generations_parts"
            generation_dir.mkdir(parents=True, exist_ok=True)

            for model_spec in MODEL_SPECS:
                print(f"\nLoading {model_spec.model_id}")
                tokenizer, model = load_text_generation_model(model_spec.model_id, quantize_4bit=QUANTIZE_4BIT)
                for decoding in DECODINGS:
                    print(f"Generating: {model_spec.model_id} / {decoding.name}")
                    rows = generation_rows(
                        prompts=prompts,
                        model_spec=model_spec,
                        decoding=decoding,
                        tokenizer=tokenizer,
                        model=model,
                        seeds=SEEDS,
                    )
                    part = pd.DataFrame(rows)
                    part_path = generation_dir / f"{model_spec.family}_{model_spec.stage}_{decoding.name}.csv"
                    save_dataframe(part, part_path)
                    all_parts.append(part)

                    combined = pd.concat(all_parts, ignore_index=True)
                    combined_path = save_dataframe(combined, Path(config["drive_data_root"]) / "ocn_generations.csv")
                    repo_url = publish_dataframe_to_hf(
                        combined,
                        repo_id=config["hf_generation_repo"],
                        split="train",
                        private=config["hf_private"],
                        card_path=REPO_ROOT / "dataset_cards/ocn_generations.md",
                        commit_message=f"Update OCN generations {config['run_id']}",
                    )
                    wandb.log({
                        "generated_rows": len(combined),
                        "last_part_rows": len(part),
                        "model_id": model_spec.model_id,
                        "decoding": decoding.name,
                    })
                    print("Autosaved:", combined_path)
                    print("Autopublished:", repo_url)

                del model, tokenizer
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            run.finish()